In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# OBTENEMOS LOS DATOS DE LA CAPA RAW
bronze_raw = spark.table("spotify_catalog.raw.spotify_api_raw")

display(bronze_raw.limit(10))

In [0]:
# CONVERTIR LA COLUMNA payload EN UN JSON CON LOS CAMPOS NECESARIOS

bronze_tracks = (
    bronze_raw
    .select(
        "search_term",
        "offset",
        "extraction_timestamp",
        from_json(
            "payload",
            """
            STRUCT<
                tracks: STRUCT<
                    items: ARRAY<STRUCT<
                        id: STRING,
                        name: STRING,
                        popularity: INT,
                        duration_ms: BIGINT,
                        explicit: BOOLEAN,
                        uri: STRING,
                        href: STRING,
                        external_urls: STRUCT<
                            spotify: STRING
                        >,
                        album: STRUCT<
                            id: STRING,
                            name: STRING,
                            album_type: STRING,
                            release_date: STRING,
                            release_date_precision: STRING,
                            total_tracks: INT,
                            href: STRING,
                            uri: STRING
                        >,
                        artists: ARRAY<STRUCT<
                            id: STRING,
                            name: STRING,
                            href: STRING,
                            uri: STRING
                        >>
                    >>
                >
            >
            """
        ).alias("data")
    )
)

display(bronze_tracks.limit(5))

In [0]:
# POR CADA ELEMENTO DE LA COLUMNA "DATA" EXISTEN 10 TRACKS.ITEMS
# SE GENERA UNA FILA POR CADA TRACK.ITEM
# 170 FILAS DE TRACKS POR 10 ITEMS POR CADA TRACKS = 1700 TRACKS.ITEMS 

bronze_tracks = (
    bronze_tracks
    .select(
        "search_term",
        "extraction_timestamp",
        explode("data.tracks.items").alias("track")
    )
)

display(bronze_tracks.limit(5))

In [0]:
bronze_tracks.count()

In [0]:
bronze_tracks = (
    bronze_tracks
    .select(
        "search_term",
        "extraction_timestamp",
        "track",
        explode("track.artists").alias("artist")
    )
)
display(bronze_tracks.limit(5))

In [0]:
bronze_tracks.count()

In [0]:
display(bronze_tracks.limit(10))

In [0]:
# DESCOMPONEMOS EL JSON EN COLUMNAS CON LOS DATOS DEL TRACK, ALBUM Y ARTIST

bronze_tracks_final = bronze_tracks.select(
    col("track.id").alias("track_id"),
    col("track.name").alias("track_name"),
    col("track.popularity").alias("popularity"),
    col("track.duration_ms").alias("duration_ms"),
    col("track.explicit").alias("explicit"),
    col("track.uri").alias("track_uri"),
    col("track.external_urls.spotify").alias("spotify_url"),

    col("track.album.id").alias("album_id"),
    col("track.album.name").alias("album_name"),
    col("track.album.album_type").alias("album_type"),
    col("track.album.release_date").alias("release_date"),
    col("track.album.total_tracks").alias("album_total_tracks"),
    col("artist.id").alias("artist_id"),
    col("artist.name").alias("artist_name"),
    col("artist.href").alias("artist_href"),
    col("artist.uri").alias("artist_uri"),
    "search_term",
    "extraction_timestamp"
)

In [0]:
display(bronze_tracks_final.limit(10))

In [0]:
bronze_tracks_final.count()

In [0]:
# COPIAMOS LA TABLA A LA CAPA BRONZE
(
    bronze_tracks_final
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(
        "spotify_catalog.bronze.spotify_tracks"
    )
)